# 부록 01. LangChain과 LangGraph 소개

이 노트북에서는 LangChain 1.0과 LangGraph의 기본 개념을 소개합니다.

## 학습 목표
- LangChain과 LangGraph의 차이점과 관계 이해
- LangChain 1.0의 주요 변경사항 파악
- 기본 환경 설정 방법 학습

## 1. LangChain이란?

**LangChain**은 LLM(Large Language Model) 기반 애플리케이션을 구축하기 위한 프레임워크입니다.

### 핵심 기능
- **모델 통합**: OpenAI, Anthropic, Google 등 다양한 LLM 제공자와의 통합
- **도구(Tools)**: 에이전트가 외부 시스템과 상호작용할 수 있는 기능 제공
- **메모리**: 대화 히스토리 및 상태 관리
- **에이전트**: 도구를 활용하여 복잡한 작업을 수행하는 자율 시스템

## 2. LangGraph란?

**LangGraph**는 LangChain 위에 구축된 그래프 기반 에이전트 오케스트레이션 프레임워크입니다.

### LangGraph의 특징
- **상태 그래프(StateGraph)**: 노드와 엣지로 구성된 워크플로우 정의
- **조건부 라우팅**: 상태에 따른 동적 실행 경로 결정
- **체크포인팅**: 실행 상태 저장 및 복원
- **서브그래프**: 모듈화된 그래프 구성

### LangChain vs LangGraph

| 구분 | LangChain | LangGraph |
|------|-----------|------------|
| 추상화 수준 | 높음 (create_agent) | 낮음 (StateGraph) |
| 유연성 | 제한적 | 매우 높음 |
| 사용 난이도 | 쉬움 | 중간 |
| 적합한 용도 | 간단한 에이전트 | 복잡한 워크플로우 |

## 3. 환경 설정

In [ ]:
# 필요한 패키지 설치 (이미 설치되어 있다면 스킵)
# !pip install langchain langchain-openai langgraph python-dotenv

In [ ]:
import os
from dotenv import load_dotenv

# .env 파일에서 환경 변수 로드
load_dotenv()

# API 키 확인
print("환경 설정 상태:")
print(f"- OPENAI_API_KEY: {'설정됨' if os.getenv('OPENAI_API_KEY') else '미설정'}")
print(f"- ANTHROPIC_API_KEY: {'설정됨' if os.getenv('ANTHROPIC_API_KEY') else '미설정'}")
print(f"- LANGSMITH_API_KEY: {'설정됨' if os.getenv('LANGSMITH_API_KEY') else '미설정'}")

## 4. LangChain 1.0 주요 변경사항

LangChain 1.0에서는 API가 크게 간소화되었습니다.

### 주요 변경점

1. **`create_agent` 함수**: 에이전트 생성이 단순화됨
2. **`init_chat_model` 함수**: 모델 초기화가 통일됨
3. **Middleware 시스템**: 에이전트 실행 흐름을 세밀하게 제어
4. **ToolRuntime**: 도구에서 상태와 컨텍스트에 접근하는 표준 방법
5. **TypedDict 기반 State**: Pydantic 대신 TypedDict 사용

## 5. 간단한 예제: Hello LangChain 1.0

In [ ]:
from langchain.chat_models import init_chat_model

# 모델 초기화 (LangChain 1.0 방식)
model = init_chat_model(
    "gpt-4o-mini",
    temperature=0
)

# 간단한 호출
response = model.invoke("안녕하세요! LangChain 1.0을 사용 중입니다.")
print(response.content)

In [ ]:
from langchain.agents import create_agent

# 간단한 도구 정의
def get_current_time() -> str:
    """현재 시간을 반환합니다."""
    from datetime import datetime
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# 에이전트 생성 (LangChain 1.0 방식)
agent = create_agent(
    model="gpt-4o-mini",
    tools=[get_current_time],
    system_prompt="당신은 도움이 되는 어시스턴트입니다."
)

# 에이전트 실행
result = agent.invoke({
    "messages": [{"role": "user", "content": "지금 몇 시야?"}]
})

print("에이전트 응답:")
print(result["messages"][-1].content)

## 6. LangGraph 기본 예제

In [ ]:
from typing import TypedDict, Annotated, Literal
import operator
from langgraph.graph import StateGraph, START, END

# 상태 정의 (TypedDict 사용)
class SimpleState(TypedDict):
    messages: Annotated[list, operator.add]
    step_count: int

# 노드 함수 정의
def process_input(state: SimpleState) -> dict:
    """입력을 처리하는 노드"""
    return {
        "messages": [f"입력 처리 완료: {state['messages'][-1]}"],
        "step_count": state.get("step_count", 0) + 1
    }

def generate_response(state: SimpleState) -> dict:
    """응답을 생성하는 노드"""
    return {
        "messages": [f"최종 응답 (스텝 {state['step_count']}): 처리 완료!"],
        "step_count": state["step_count"] + 1
    }

# 그래프 구성
graph = StateGraph(SimpleState)

# 노드 추가
graph.add_node("process", process_input)
graph.add_node("respond", generate_response)

# 엣지 추가
graph.add_edge(START, "process")
graph.add_edge("process", "respond")
graph.add_edge("respond", END)

# 그래프 컴파일
app = graph.compile()

# 실행
result = app.invoke({"messages": ["Hello LangGraph!"], "step_count": 0})
print("결과:")
for msg in result["messages"]:
    print(f"  - {msg}")

## 7. 요약

### 이 노트북에서 배운 것

1. **LangChain**: LLM 애플리케이션 구축을 위한 프레임워크
   - 모델 통합, 도구, 메모리, 에이전트 기능 제공

2. **LangGraph**: 그래프 기반 에이전트 오케스트레이션
   - StateGraph로 복잡한 워크플로우 정의
   - 더 낮은 수준의 제어 가능

3. **LangChain 1.0 주요 기능**:
   - `create_agent`: 간단한 에이전트 생성
   - `init_chat_model`: 통합된 모델 초기화
   - Middleware: 실행 흐름 제어
   - ToolRuntime: 상태/컨텍스트 접근

### 다음 단계
- 부록_02: `init_chat_model`로 다양한 모델 초기화하기
- 부록_03: Tool 데코레이터와 ToolRuntime 활용하기